# M2 — Watermarks: scenario testing

M2-only isolation: every verdict in this notebook is caused exclusively by
watermark evidence. Provenance, forensics, and the detector are silenced.

Module scope: detects the DWT-DCT blind watermark scheme used by Stable
Diffusion pipelines (two known payloads: SDXL 48-bit, SD-v1 text). All other
deployed schemes (SynthID, Meta, Stable Signature) are closed or per-model-keyed
— documented, not detected.

Convention: commit WITH outputs — this is a run record.

In [ ]:
# ── Setup — fresh-kernel-proof ──────────────────────────────────
!apt-get -qq install -y libimage-exiftool-perl > /dev/null
!git clone -q https://github.com/Waranika/AI-image-Checkers.git 2>/dev/null || echo "already cloned"
%cd /content/AI-image-Checkers
%pip install -q -e .

import numpy as np
from pathlib import Path
from PIL import Image
from ai_image_id.ingest import ingest
from ai_image_id.watermark import analyze_watermarks, dwt_dct, SDXL_BITS, SD_V1_BITS, KNOWN_PAYLOADS
from ai_image_id.fusion import fuse
from ai_image_id.schema import Evidence, ProvenanceEvidence, SpectrumEvidence

def report(path, label=""):
    """M2-only evidence card. No provenance, no FFT, no detector."""
    img = ingest(path)
    wms = analyze_watermarks(img.rgb)
    evidence = Evidence(
        provenance=ProvenanceEvidence(),         # M1 silenced
        watermarks=wms,
        spectrum=SpectrumEvidence(valid=False),   # M3 silenced
        detector=None,                            # M4 silenced
    )
    r = fuse(evidence, sha256=img.sha256, phash=img.phash)
    best = next((w for w in wms if w.scheme == "dwtDct" and w.bit_accuracy is not None), None)
    print(f"┌─ {label or path}")
    print(f"│ verdict: {r.ai_verdict.value} ({r.confidence})")
    if best:
        print(f"│ dwtDct: detected={best.detected}  bit_acc={best.bit_accuracy}  "
              f"payload={best.matched_payload}")
    for w in wms:
        if not w.applicable:
            print(f"│ {w.scheme}: not applicable — {w.notes}")
    print(f"└ notes: {r.notes}\n")
    return r, best

def bit_accuracy(path):
    """Quick bit-accuracy only, for sweep tables."""
    img = ingest(path)
    wms = analyze_watermarks(img.rgb)
    best = next((w for w in wms if w.scheme == "dwtDct" and w.bit_accuracy is not None), None)
    return best.bit_accuracy if best else None

print("setup OK — SDXL payload:", len(SDXL_BITS), "bits | SD-v1 payload:", len(SD_V1_BITS), "bits")

## 1 — Synthetic round-trip (controlled ground truth)

Embed known payloads into a synthetic image, decode them. This tests the
codec's self-consistency, NOT whether it reads what real SD writes (that's §3).
Lossless save preserves the watermark; lossy attacks degrade it.

In [ ]:
FIX = Path("/content/fixtures_m2"); FIX.mkdir(exist_ok=True)

rng = np.random.default_rng(7)
y, x = np.mgrid[0:512, 0:512]
base = np.clip(np.stack([120+60*np.sin(x/97), 100+50*np.cos(y/83),
                         90+40*np.sin((x+y)/71)], -1) + rng.normal(0,6,(512,512,3)),
               0, 255).astype(np.uint8)

# a) Lossless: should detect → verified
marked = dwt_dct.embed(base, SDXL_BITS)
p = FIX/"sdxl_lossless.png"; Image.fromarray(marked).save(p)
report(str(p), "SDXL payload, lossless PNG")

# b) High-quality JPEG: should still detect
p = FIX/"sdxl_q92.jpg"; Image.fromarray(marked).save(p, quality=92)
report(str(p), "SDXL payload, Q92 JPEG")

# c) Low-quality JPEG: expected to fail — the watermark should be destroyed
p = FIX/"sdxl_q70.jpg"; Image.fromarray(marked).save(p, quality=70)
report(str(p), "SDXL payload, Q70 JPEG")

# d) SD-v1 payload, lossless
marked_v1 = dwt_dct.embed(base, SD_V1_BITS)
p = FIX/"sdv1_lossless.png"; Image.fromarray(marked_v1).save(p)
report(str(p), "SD-v1 payload, lossless PNG")

# e) Clean image — no watermark: should be inconclusive
p = FIX/"clean.png"; Image.fromarray(base).save(p)
report(str(p), "no watermark (baseline)")

## 2 — Fragility curve: bit accuracy vs. JPEG quality

The cliff between "detectable" and "destroyed" tells us exactly where M2
stops being useful on real-world images (most platforms recompress at Q75–85).
Threshold line at 0.90 = our detection cutoff.

In [ ]:
import matplotlib.pyplot as plt

marked = dwt_dct.embed(base, SDXL_BITS)
qualities = list(range(50, 100, 2))
accuracies = []
for q in qualities:
    p = FIX/f"sweep_q{q}.jpg"
    Image.fromarray(marked).save(p, quality=q)
    accuracies.append(bit_accuracy(str(p)))

plt.figure(figsize=(10, 4))
plt.plot(qualities, accuracies, "o-", markersize=4)
plt.axhline(0.90, color="red", linestyle="--", label="detection threshold (0.90)")
plt.axhline(0.50, color="gray", linestyle=":", label="coin flip (0.50)")
plt.xlabel("JPEG quality"); plt.ylabel("bit accuracy")
plt.title("DWT-DCT watermark survival vs. JPEG quality")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig("/content/m2_jpeg_curve.png", dpi=150)
plt.show()

# Find the cliff
cliff = next((q for q, a in zip(qualities, accuracies) if a and a >= 0.90), None)
print(f"watermark detectable (≥0.90) above Q={cliff}" if cliff else "never reaches threshold")
print(f"Q70={accuracies[qualities.index(70)]:.3f}  Q80={accuracies[qualities.index(80)]:.3f}  "
      f"Q92={accuracies[qualities.index(92)]:.3f}")

## 3 — Transport-degradation matrix (M2 column)

Same hops as notebook 03 (M1). The hypothesis: watermarks survive transforms
that kill metadata. If true, watermarks and provenance are complementary
(M1 proves origin on pristine images; M2 extends detection through mild
reprocessing). If false, M2 adds little over M1.

In [ ]:
import shutil, subprocess

up_path = None
try:
    from google.colab import files
    up = files.upload()
    up_path = Path(next(iter(up)))
except Exception:
    pass

# Use the synthetic watermarked image if no upload
orig = up_path if up_path else Path(FIX/"sdxl_lossless.png")
HOP = Path("/content/hops_m2"); HOP.mkdir(exist_ok=True)
im = Image.open(orig).convert("RGB")

hops = {"0-original": orig}
p = HOP/"1-resave.jpg";     im.save(p, quality=92);  hops["1-resave-jpg"] = p
p = HOP/"2-screenshot.png"; im.save(p);               hops["2-screenshot"] = p
p = HOP/"3-messenger.jpg";  im.resize((im.width//2, im.height//2)).save(p, quality=70)
hops["3-messenger"] = p
p = HOP/"4-strip"+orig.suffix; shutil.copy(orig, p)
subprocess.run(["exiftool","-overwrite_original","-all=",str(p)], capture_output=True)
hops["4-exiftool-strip"] = p
p = HOP/"5-crop.jpg"; im.crop((50,50,im.width-50,im.height-50)).save(p, quality=92)
hops["5-crop"] = p
p = HOP/"6-pil-reencode.png"; im.save(p)
hops["6-pil-reencode"] = p

print(f"{'transport':<20} {'bit_acc':<10} {'detected':<10} verdict")
for name, path in hops.items():
    r, best = report.__wrapped__(str(path), name) if hasattr(report, '__wrapped__') else (None, None)
    # inline version:
    img = ingest(path)
    wms = analyze_watermarks(img.rgb)
    best = next((w for w in wms if w.scheme=="dwtDct" and w.bit_accuracy is not None), None)
    evidence = Evidence(provenance=ProvenanceEvidence(), watermarks=wms,
                        spectrum=SpectrumEvidence(valid=False), detector=None)
    r = fuse(evidence, sha256=img.sha256, phash=img.phash)
    ba = f"{best.bit_accuracy:.3f}" if best else "n/a"
    det = str(best.detected) if best else "n/a"
    print(f"{name:<20} {ba:<10} {det:<10} {r.ai_verdict.value}")

### Reading the matrix

Compare with the M1 matrix from notebook 03. Expected pattern:
- **M1 (provenance):** dies at every hop except exiftool-strip (C2PA survives)
- **M2 (watermark):** survives hops that don't heavily recompress — re-save Q92,
  screenshot (lossless PNG re-encode), exiftool-strip, PIL re-encode. Dies at
  messenger (Q70 + resize) and possibly crop.

If the patterns are complementary, the two modules together cover more
scenarios than either alone — which is the evidence-fusion thesis.

If the watermark dies everywhere M1 also dies, M2 adds nothing in practice.

## 4 — Real SDXL image validation (the oldest open item)

The vendored DWT-DCT decoder has never met a real SD image. Generate one in
this session (or upload one from civitai/an A1111 instance). This answers:
does our decoder read what real Stable Diffusion actually writes?

**Option A** — generate in-session (needs GPU runtime + ~5 min download):

In [ ]:
# Uncomment to generate a real SDXL image with the official watermark:
# %pip install -q diffusers transformers accelerate invisible-watermark
# import torch
# from diffusers import StableDiffusionXLPipeline
#
# pipe = StableDiffusionXLPipeline.from_pretrained(
#     "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16
# ).to("cuda")
# img = pipe("a castle on a green hill, fantasy art", num_inference_steps=20).images[0]
# img.save("/content/real_sdxl.png")
# report("/content/real_sdxl.png", "real SDXL output")

# Option B — upload a real SD image:
# from google.colab import files
# up = files.upload()
# for name in up:
#     report(name, f"uploaded: {name}")

## 5 — Scope documentation

What M2 detects, what it can't, and why — reference for the README/blog.

| scheme | detectable? | status |
|---|---|---|
| DWT-DCT (SD 1.x/2.x/SDXL default) | ✓ (known payloads) | implemented; real-SD validation pending |
| Stable Signature (Meta/ICCV'23) | partial | decoder public, keys per-model — can decode bits, can't verify |
| SynthID (Google/DeepMind) | ✗ | closed detector; M1 can report C2PA declaration of its presence |
| Meta invisible watermark | ✗ | no public decoder |
| Tree-Ring | ✗ (on arbitrary images) | requires injection-time key; demo-only for self-generated images |

The ecosystem is mostly closed. M2's honest capability is one scheme with
measured fragility. Everything else requires the generator's own verification
tools — which is itself a finding worth reporting.